# W24 · Gazebo Harmonic 仿真与 ros_gz 集成

> 阶段一你在 Gymnasium 里仿真「理想化动力学」；这一讲换成**全物理仿真**：
> 刚体碰撞、摩擦、惯性张量、传感器噪声模型。Gazebo 是 ROS 生态的标准答案。

## 学习目标

1. 说清 Gazebo Harmonic 的架构（`gz-sim` + Gazebo Transport）以及它与 ROS2 的关系；
2. 掌握 **Jazzy ↔ Harmonic** 官方配对与 `ros-jazzy-ros-gz` 安装；
3. 区分 **URDF（机器人描述）与 SDF（仿真场景描述）**，会把 URDF 机器人 spawn 进 Gazebo；
4. 会用 **ros_gz_bridge** 桥接话题（cmd_vel / odom / scan / imu / clock）；
5. 会在 URDF 中通过 `<gazebo>` 标签挂传感器插件。

## ⚠️ 运行前提

本讲所有 `gz` / `ros2` 命令需 ROS2 Jazzy + Gazebo Harmonic（见 `docs/phase3_ros2.md`）。
本机可执行的部分：用 Python 解析/生成 SDF 与桥接配置（标准库 + PyYAML）。

## 1. 架构：两个世界，一座桥

先建立最重要的心智模型：**Gazebo 和 ROS2 是两个独立的进程世界**。

```
┌────────────────────────────┐        ┌──────────────────────────┐
│   Gazebo (gz-sim)          │        │   ROS2 (DDS)             │
│  - 物理引擎 (DART)          │        │  - 你的节点 (rclpy)       │
│  - 传感器模拟 (lidar/imu)   │  桥     │  - Nav2 / ros2_control    │
│  - Gazebo Transport 话题    │ ══════ │  - DDS 话题               │
└────────────────────────────┘ ros_gz └──────────────────────────┘
```

- **Gazebo Transport**：Gazebo 自己的 pub/sub 中间件（基于 ZeroMQ + Protobuf），
  话题名形如 `/world/empty/model/diffbot/cmd_vel`；
- **ros_gz_bridge**：一个翻译节点，把指定话题在两个中间件之间双向/单向搬运；
- **版本配对**（[官方表格](https://gazebosim.org/docs/harmonic/ros_installation)）：
  **ROS2 Jazzy 官方配对 Gazebo Harmonic（均为 LTS）**。安装一条命令：

```bash
sudo apt install ros-jazzy-ros-gz
```

> 为什么强调配对：ros_gz 对 Gazebo 库有 ABI 依赖，混装（如 Jazzy + Fortress）
> 要么装不上、要么运行时段错误——初学者不要尝试非默认组合。

## 2. 快速体验：空世界 + 内置模型

```bash
# 启动空世界（GUI）
gz sim empty.sdf

# 列出 Gazebo Transport 话题（注意：不是 ros2 topic！）
gz topic -l

# 发布一条 Gazebo 侧的速度指令（Twist 是 protobuf 类型 gz.msgs.Twist）
gz topic -t "/model/vehicle_blue/cmd_vel" -m gz.msgs.Twist -p "linear: {x: 1.0}"

# 下载并运行带传感器的示例世界（首次会从 fuel.gazebosim.org 拉模型）
gz sim sensors.sdf
```

**预期**：GUI 出现地面与重力；`gz topic -l` 打印 `/clock`、`/stats`、`/world/empty/...` 等。
此时 `ros2 topic list` **看不到**任何 Gazebo 话题——桥还没架，两个世界是隔离的。

## 3. SDF vs URDF：职责不同，别混用

| | URDF | SDF |
|---|---|---|
| 设计目的 | 描述**一台机器人**（树状结构） | 描述**整个仿真世界**（多机器人、灯光、物理参数、场景图，支持闭链） |
| 用在哪 | RViz、robot_state_publisher、ros2_control、MoveIt | Gazebo 加载 world 与模型 |
| 格式要点 | `<robot><link><joint>` | `<sdf><world><model><include>` |

工程惯例（也是 Nav2/ros2_control 官方推荐）：**机器人本体用 URDF/xacro 写一份**，
通过 `<gazebo>` 扩展标签附加仿真专属信息（插件、摩擦系数、传感器）；
**场景/世界用 SDF 写**。Gazebo 启动时用 `ros_gz_sim` 把 URDF 转换成内部 SDF 再 spawn。

下面用 Python 生成一个最小 world SDF，看清单个元素的职责（纯 Python，本机执行）：

In [1]:
"""解析并理解一个最小 Gazebo world SDF（纯 Python，本机执行）。"""
import xml.etree.ElementTree as ET

WORLD_SDF = """<?xml version="1.0"?>
<sdf version="1.11">
  <world name="demo_world">
    <!-- 物理引擎参数：步长 1 ms（1 kHz 物理循环） -->
    <physics name="1ms" type="ignored">
      <max_step_size>0.001</max_step_size>
      <real_time_factor>1.0</real_time_factor>
    </physics>
    <!-- 必备系统插件：物理、场景广播、用户指令 -->
    <plugin filename="gz-sim-physics-system" name="gz::sim::systems::Physics"/>
    <plugin filename="gz-sim-scene-broadcaster-system" name="gz::sim::systems::SceneBroadcaster"/>
    <plugin filename="gz-sim-user-commands-system" name="gz::sim::systems::UserCommands"/>
    <!-- 光照与大地 -->
    <light type="directional" name="sun">
      <cast_shadows>true</cast_shadows>
      <diffuse>0.8 0.8 0.8 1</diffuse>
    </light>
    <model name="ground_plane">
      <static>true</static>
      <link name="link">
        <collision name="c"><geometry><plane><normal>0 0 1</normal><size>100 100</size></plane></geometry></collision>
        <visual name="v"><geometry><plane><normal>0 0 1</normal><size>100 100</size></plane></geometry></visual>
      </link>
    </model>
  </world>
</sdf>
"""

root = ET.fromstring(WORLD_SDF)
world = root.find("world")
print("world 名称:", world.get("name"))
step = float(world.find("physics/max_step_size").text)
rtf = float(world.find("physics/real_time_factor").text)
print(f"物理步长: {step*1000} ms → 物理频率 {1/step:.0f} Hz; 实时倍率 {rtf}")
print("系统插件:", [p.get("filename") for p in world.findall("plugin")])
print("模型:", [m.get("name") for m in world.findall("model")])

# 思考题（练习 1 要用）：RL 环境的 dt=0.02s（50Hz 控制）意味着每步控制之间物理积分几步？
control_dt = 0.02
print(f"\n若控制周期 {control_dt}s，则每个控制步内物理积分 {control_dt/step:.0f} 步")

world 名称: demo_world
物理步长: 1.0 ms → 物理频率 1000 Hz; 实时倍率 1.0
系统插件: ['gz-sim-physics-system', 'gz-sim-scene-broadcaster-system', 'gz-sim-user-commands-system']
模型: ['ground_plane']

若控制周期 0.02s，则每个控制步内物理积分 20 步


**观察**：`max_step_size=0.001` 意味着 Gazebo 内部以 1 kHz 积分物理。
如果你的策略以 50 Hz 输出动作，每个控制周期之间世界已经演化了 20 个物理步——
这正是「动作延迟/部分可观测」等 Sim2Real 问题的物理来源（回顾 W16）。

## 4. 把 URDF 机器人 spawn 进 Gazebo

假设你有 W23 的 `diffbot.urdf`。标准流程（在 ROS2 机器上）：

```bash
# 终端 1：启动 Gazebo + 桥（ros_gz_sim 提供 launch）
ros2 launch ros_gz_sim gz_sim.launch.py gz_args:="empty.sdf"

# 终端 2：把 URDF 作为实体 spawn 进去（-string 直接传内容；也可 -file）
ros2 run ros_gz_sim create -string "$(cat diffbot.urdf)" -name diffbot -z 0.1

# 终端 3：架桥——把 Gazebo 话题映射成 ROS2 话题
ros2 run ros_gz_bridge parameter_bridge \
  /clock@rosgraph_msgs/msg/Clock[gz.msgs.Clock \
  /cmd_vel@geometry_msgs/msg/Twist]gz.msgs.Twist \
  /model/diffbot/odometry@nav_msgs/msg/Odometry[gz.msgs.Odometry
```

桥接语法 `@消息类型[` 表示 **Gazebo→ROS**，`]` 表示 **ROS→Gazebo**，`@` 表示双向。

**预期**：`ros2 topic list` 出现 `/clock /cmd_vel /model/diffbot/odometry`；
向 `/cmd_vel` 发 Twist，Gazebo 中的小车开始移动（需先配好 W25 的控制器或
Gazebo 的 `diff_drive` 系统插件，见下一节）。

## 5. 在 URDF 里挂传感器：`<gazebo>` 标签 + 系统插件

Gazebo 的传感器由**系统插件**驱动。在 URDF 中用 `<gazebo>` 标签声明，例如 2D 激光雷达：

```xml
<link name="lidar_link">
  <visual><geometry><cylinder radius="0.05" length="0.05"/></geometry></visual>
</link>
<joint name="lidar_joint" type="fixed">
  <parent link="base_link"/><child link="lidar_link"/>
  <origin xyz="0.15 0 0.12"/>
</joint>

<gazebo reference="lidar_link">
  <sensor name="lidar" type="gpu_lidar">
    <update_rate>10</update_rate>
    <lidar><scan>
      <horizontal><samples>360</samples><min_angle>-3.14159</min_angle><max_angle>3.14159</max_angle></horizontal>
    </scan>
    <range><min>0.1</min><max>12.0</max></range></lidar>
    <always_on>true</always_on>
  </sensor>
</gazebo>
```

同时世界（或模型）需加载传感器系统插件：`gz-sim-sensors-system`（渲染类传感器）
——`ros_gz_sim` 的默认 launch 已带上常用集合。

桥接点云/激光时**注意 QoS**：Gazebo 侧传感器是 `best_effort`，ROS2 端订阅必须匹配：

```bash
ros2 run ros_gz_bridge parameter_bridge /scan@sensor_msgs/msg/LaserScan[gz.msgs.LaserScan
ros2 topic echo /scan --no-arr        # 若收不到，先检查 QoS 兼容性（W21 的坑）
```

## 6. 桥接配置工程化：YAML 一次声明

话题一多，命令行参数不可维护。`ros_gz_bridge` 支持 YAML 配置（`config.yaml`）：

```yaml
- ros_topic_name: "/clock"
  gz_topic_name: "/clock"
  ros_type_name: "rosgraph_msgs/msg/Clock"
  gz_type_name: "gz.msgs.Clock"
  direction: GZ_TO_ROS
- ros_topic_name: "/cmd_vel"
  gz_topic_name: "/model/diffbot/cmd_vel"
  ros_type_name: "geometry_msgs/msg/Twist"
  gz_type_name: "gz.msgs.Twist"
  direction: ROS_TO_GZ
```

启动：`ros2 run ros_gz_bridge parameter_bridge --ros-args -p config_file:=config.yaml`。
下面用 Python 从「话题清单」自动生成该 YAML（纯 Python，本机执行）——
这正是工程里「配置即代码」的思路：

In [2]:
"""从话题清单自动生成 ros_gz_bridge 的 YAML 配置（纯 Python，本机执行）。"""
import yaml

BRIDGE_TOPICS = [
    # (ros 话题, gz 话题, ros 类型, gz 类型, 方向)
    ("/clock", "/clock", "rosgraph_msgs/msg/Clock", "gz.msgs.Clock", "GZ_TO_ROS"),
    ("/cmd_vel", "/model/diffbot/cmd_vel", "geometry_msgs/msg/Twist", "gz.msgs.Twist", "ROS_TO_GZ"),
    ("/odom", "/model/diffbot/odometry", "nav_msgs/msg/Odometry", "gz.msgs.Odometry", "GZ_TO_ROS"),
    ("/scan", "/world/demo_world/model/diffbot/link/lidar_link/sensor/lidar/scan",
     "sensor_msgs/msg/LaserScan", "gz.msgs.LaserScan", "GZ_TO_ROS"),
    ("/imu", "/imu", "sensor_msgs/msg/Imu", "gz.msgs.IMU", "GZ_TO_ROS"),
]

config = [
    {
        "ros_topic_name": r,
        "gz_topic_name": g,
        "ros_type_name": rt,
        "gz_type_name": gt,
        "direction": d,
    }
    for r, g, rt, gt, d in BRIDGE_TOPICS
]

text = yaml.safe_dump(config, sort_keys=False, allow_unicode=True)
print(text)
print(f"共 {len(config)} 条桥接规则；ROS→GZ {sum(1 for *_, d in BRIDGE_TOPICS if d=='ROS_TO_GZ')} 条，"
      f"GZ→ROS {sum(1 for *_, d in BRIDGE_TOPICS if d=='GZ_TO_ROS')} 条")

- ros_topic_name: /clock
  gz_topic_name: /clock
  ros_type_name: rosgraph_msgs/msg/Clock
  gz_type_name: gz.msgs.Clock
  direction: GZ_TO_ROS
- ros_topic_name: /cmd_vel
  gz_topic_name: /model/diffbot/cmd_vel
  ros_type_name: geometry_msgs/msg/Twist
  gz_type_name: gz.msgs.Twist
  direction: ROS_TO_GZ
- ros_topic_name: /odom
  gz_topic_name: /model/diffbot/odometry
  ros_type_name: nav_msgs/msg/Odometry
  gz_type_name: gz.msgs.Odometry
  direction: GZ_TO_ROS
- ros_topic_name: /scan
  gz_topic_name: /world/demo_world/model/diffbot/link/lidar_link/sensor/lidar/scan
  ros_type_name: sensor_msgs/msg/LaserScan
  gz_type_name: gz.msgs.LaserScan
  direction: GZ_TO_ROS
- ros_topic_name: /imu
  gz_topic_name: /imu
  ros_type_name: sensor_msgs/msg/Imu
  gz_type_name: gz.msgs.IMU
  direction: GZ_TO_ROS

共 5 条桥接规则；ROS→GZ 1 条，GZ→ROS 4 条


## 7. 仿真时钟：`/clock` 与 `use_sim_time`

Gazebo 可以被暂停、可以慢于/快于实时运行。如果节点用系统时间，暂停仿真时
TF 和控制器会全线错乱。约定：**所有节点设置参数 `use_sim_time:=true`**，
统一使用桥接过来的 `/clock`。

```bash
ros2 run my_pkg my_node --ros-args -p use_sim_time:=true
```

> 排查口诀：TF 报「时间过期/未来」、定时器不走——九成是 `use_sim_time` 没设或 `/clock` 没桥。

## ✏️ 练习

### 练习 1（★，约 15 分钟，纯 Python）：控制频率与物理步

基于第 3 节的 SDF 解析代码，写一个函数 `warn_if_misaligned(step_size, control_dt)`：
当 `control_dt` 不是 `step_size` 的整数倍时警告（会造成动作作用时间不匀）。
测试 `(0.001, 0.02)`、`(0.001, 0.015)`、`(0.004, 0.02)`。交付：函数 + 测试输出 +
一句话结论（为什么阶段一的 `dt=0.02` 在 Gazebo 里要对齐）。

### 练习 2（★★，约 40 分钟）：小车 + 激光雷达进 Gazebo

在 ROS2 机器上完成：W23 的 diffbot URDF + 第 5 节的 lidar `<gazebo>` 配置 +
spawn + 桥接 `/scan`。手动用 `gz topic` 推一个箱子到机器人面前，
观察 `ros2 topic echo /scan --no-arr` 的 `ranges` 变化。
交付：完整 URDF、桥接命令、`ranges` 最小值随箱子距离变化的 3 组数据。

### 练习 3（★★，约 25 分钟）：QoS 故障排查报告

故意制造一次 QoS 不匹配：写一个 `reliable` 订阅者订阅 Gazebo 桥接的 `/scan`（best_effort），
记录现象；然后改成匹配 QoS 修复。交付：故障现象描述（`ros2 topic info -v /scan` 的两端 QoS 输出）
+ 修复代码片段 + 一句话总结匹配规则。

### 练习 4（★★★，约 45 分钟）：相机 + AprilTag 世界

搭一个含 `camera` 传感器的世界（参考 `gz sim sensors.sdf`），桥接
`/camera/image_raw`（`sensor_msgs/msg/Image` / `gz.msgs.Image`），
用 `ros2 run rqt_image_view rqt_image_view` 看图。进阶：用 OpenCV 写个订阅节点检测颜色块中心。
交付：world/URDF 改动 diff、桥接 YAML、截图描述或检测节点的日志。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**：

```python
def warn_if_misaligned(step_size, control_dt):
    n = control_dt / step_size
    if abs(n - round(n)) > 1e-9:
        print(f"警告: control_dt={control_dt} 不是 step_size={step_size} 的整数倍 (n={n:.3f})")
    else:
        print(f"OK: 每个控制步 = {int(round(n))} 个物理步")
```

结论：动作在物理时间轴上的作用区间必须对齐整数个物理步，否则同一动作有时长有时短，
等于给策略注入了隐式噪声——Gymnasium 环境的 `dt` 是精确离散的，仿真迁移时这层抽象会破。

**练习 2**：关键在于 `<gazebo reference="lidar_link">` 中的 `<sensor type="gpu_lidar">`
与世界里加载 `gz-sim-sensors-system`；桥接后 `ranges[0]`（正前方）随箱子距离线性变化，
`min(ranges)` 应接近箱子实际距离（误差几个 cm 属正常，因为射线离散）。

**练习 3**：现象——订阅者收不到消息但 `ros2 topic hz` 显示有数据；
`ros2 topic info -v /scan` 显示 Publisher `Reliability: BEST_EFFORT`、
Subscription `Reliability: RELIABLE`，末尾标注 `QoS incompatible`。
修复：订阅端 `qos_profile_sensor_data` 或显式 `reliability=BEST_EFFORT`。
规则：订阅端的 reliability 不能高于发布端（best_effort 发布无法满足 reliable 订阅）。

**练习 4**：相机传感器类型 `camera`，gz 话题形如
`/world/.../model/.../link/camera_link/sensor/camera/image`；
桥接后 `rqt_image_view` 选 `/camera/image_raw`。
颜色检测：订阅后 `cv_bridge` 转 numpy（或手工 `np.frombuffer(msg.data, np.uint8).reshape(h, w, 3)`），
HSV 阈值 + 矩求质心。
</details>

## 延伸阅读

- [Gazebo Harmonic 官方文档](https://gazebosim.org/docs/harmonic) 与 [ROS 安装/配对指南](https://gazebosim.org/docs/harmonic/ros_installation)
- [ros_gz 仓库（含 bridge 与示例）](https://github.com/gazebosim/ros_gz)
- [Gazebo 传感器教程](https://gazebosim.org/docs/harmonic/sensors)
- [Nav2 的现代 Gazebo Getting Started（与 W26 衔接）](https://docs.nav2.org/getting_started/index.html)
- 下一讲预告：W25 用 ros2_control 给仿真机器人装上「标准化关节接口」。